# 02 - Annotation & Normalisation du Texte

**Projet :** Système de Synthèse Vocale pour l'Arabe Hassaniya par Apprentissage par Transfert
**Auteur :** Mohamed Salem Ebnou Echvagha Oubeid (C34613)
**Module :** Dialectes NLP — Master M1 IA
**Date :** Juin 2026

---

## Objectif

L'annotation du texte est une étape cruciale dans tout pipeline TTS. Avant de fournir du texte à un
modèle de synthèse vocale, nous devons :

1. **Normaliser** le texte (standardiser les formes des caractères)
2. **Nettoyer** le texte (supprimer le bruit, corriger les problèmes d'encodage)
3. **Tokeniser** au niveau des caractères (les modèles TTS opèrent sur des caractères ou des phonèmes)
4. **Construire un vocabulaire** (associer les caractères à des identifiants entiers)

### Pourquoi est-ce important pour le Hassaniya ?

L'arabe hassaniya n'a **pas d'orthographe standardisée**. Le même mot peut être écrit
de plusieurs façons. Par exemple, la lettre Alef peut apparaître sous les formes : ا, أ, إ, آ.
Nous devons normaliser ces variantes pour que le modèle TTS reçoive une entrée cohérente.

## 1. Configuration

In [ ]:
import pandas as pd
import numpy as np
import re
import unicodedata
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['figure.figsize'] = (12, 4)
sns.set_style('whitegrid')

print('Configuration terminée')

In [ ]:
# Charger le jeu de données
df = pd.read_parquet('../audios_dataset.parquet')
print(f'{len(df)} échantillons chargés')
print(f'\n5 premières transcriptions :')
for i in range(5):
    print(f'  [{i}] {df["transcription"].iloc[i]}')

## 2. Normalisation du Texte

### Qu'est-ce que la normalisation ?

La normalisation réduit le nombre de caractères uniques en associant les variantes
à une forme canonique. C'est particulièrement important pour l'arabe car :

| Original | Normalisé | Raison |
|----------|-----------|--------|
| أ, إ, آ, ٱ | ا | Toutes les formes d'Alef → Alef de base |
| ؤ | و | Hamza sur Waw → Waw simple |
| ئ | ي | Hamza sur Ya → Ya simple |
| ة | ه | Ta Marbuta → Ha |
| ى | ي | Alef Maqsura → Ya |

Nous **supprimons également les diacritiques** (tashkeel) car notre jeu de données ne les utilise
pas de manière cohérente, et ils créeraient de la rareté dans les données.

In [ ]:
class HassaniyaTextProcessor:
    """Processeur de texte adapté au dialecte arabe hassaniya."""
    
    # Regex pour les signes diacritiques arabes (tashkeel)
    ARABIC_DIACRITICS = re.compile(r'[ً-ٰ۟]')
    
    # Table de normalisation des caractères
    CHAR_NORMALIZATIONS = {
        'آ': 'ا',  # آ → ا
        'أ': 'ا',  # أ → ا
        'إ': 'ا',  # إ → ا
        'ٱ': 'ا',  # ٱ → ا
        'ؤ': 'و',  # ؤ → و
        'ئ': 'ي',  # ئ → ي
        'ة': 'ه',  # ة → ه
        'ى': 'ي',  # ى → ي
        'ٹ': 'ت',  # ٹ → ت
        'ڤ': 'ف',  # ڤ → ف
        'گ': 'ك',  # گ → ك
    }
    
    def normalize(self, text):
        """Appliquer le pipeline complet de normalisation."""
        # Étape 1 : Normalisation Unicode NFKC
        text = unicodedata.normalize('NFKC', text)
        # Étape 2 : Suppression des diacritiques
        text = self.ARABIC_DIACRITICS.sub('', text)
        # Étape 3 : Normalisation des variantes de caractères
        for src, dst in self.CHAR_NORMALIZATIONS.items():
            text = text.replace(src, dst)
        # Étape 4 : Suppression des caractères non-arabes (garder arabe + espaces + ponctuation de base)
        text = re.sub(r'[^؀-ۿ\s.,،؛:؟!؟\-]', '', text)
        # Étape 5 : Normalisation des espaces
        text = re.sub(r'\s+', ' ', text).strip()
        return text
    
    def char_tokenize(self, text):
        """Tokenisation au niveau des caractères avec marqueurs d'espace."""
        tokens = []
        for ch in text:
            if ch == ' ':
                tokens.append('<space>')
            else:
                tokens.append(ch)
        return tokens
    
    def build_vocab(self, texts):
        """Construire le vocabulaire de caractères à partir d'une liste de textes."""
        chars = set()
        for text in texts:
            normalized = self.normalize(text)
            chars.update(normalized)
        
        # Tokens spéciaux d'abord, puis caractères triés
        vocab = {'<pad>': 0, '<sos>': 1, '<eos>': 2, '<space>': 3}
        for i, ch in enumerate(sorted(chars - {' '}), start=4):
            vocab[ch] = i
        return vocab
    
    def text_to_ids(self, text, vocab):
        """Convertir le texte en séquence d'identifiants entiers."""
        normalized = self.normalize(text)
        tokens = self.char_tokenize(normalized)
        return [vocab.get(t, 0) for t in tokens]


processor = HassaniyaTextProcessor()
print('Processeur initialisé')

In [ ]:
# Démonstration de la normalisation
print('=== Exemples de Normalisation ===')
print()
for i in range(min(8, len(df))):
    original = df['transcription'].iloc[i]
    normalized = processor.normalize(original)
    changed = original != normalized
    marker = ' [MODIFIÉ]' if changed else ''
    print(f'Original :   {original}')
    print(f'Normalisé :  {normalized}{marker}')
    print()

In [ ]:
# Appliquer la normalisation à l'ensemble du jeu de données
df['normalized_text'] = df['transcription'].apply(processor.normalize)

# Compter combien de textes ont changé
changed_count = (df['transcription'] != df['normalized_text']).sum()
print(f'Textes modifiés par la normalisation : {changed_count} / {len(df)} ({100*changed_count/len(df):.1f}%)')

# Comparer le nombre de caractères avant et après
original_chars = set(''.join(df['transcription'].tolist())) - {' '}
normalized_chars = set(''.join(df['normalized_text'].tolist())) - {' '}

print(f'\nCaractères uniques avant normalisation : {len(original_chars)}')
print(f'Caractères uniques après normalisation :  {len(normalized_chars)}')
print(f'Caractères supprimés/fusionnés : {len(original_chars) - len(normalized_chars)}')

## 3. Tokenisation au Niveau des Caractères

### Pourquoi au niveau des caractères ?

Les modèles TTS fonctionnent généralement au **niveau des caractères** ou au **niveau des phonèmes** :

- **Niveau caractères** : Chaque lettre arabe devient un token. Simple et efficace pour l'arabe.
- **Niveau phonèmes** : Chaque unité sonore devient un token. Plus précis mais nécessite un dictionnaire de phonèmes.

Pour notre MVP, nous utilisons la **tokenisation au niveau des caractères** car :
1. Aucun dictionnaire de phonèmes hassaniya n'existe
2. L'écriture arabe est largement phonétique — les lettres correspondent étroitement aux sons
3. Plus simple à implémenter avec nos contraintes de temps

In [ ]:
# Construire le vocabulaire
vocab = processor.build_vocab(df['transcription'].tolist())

print(f'Taille du vocabulaire : {len(vocab)}')
print(f'\nTable de correspondance du vocabulaire :')
for token, idx in sorted(vocab.items(), key=lambda x: x[1]):
    display_token = token if token.startswith('<') else f'  {token}  '
    print(f'  {idx:3d} → {display_token}')

In [ ]:
# Démonstration de la tokenisation
print('=== Exemples de Tokenisation ===')
print()
for i in range(3):
    text = df['transcription'].iloc[i]
    normalized = processor.normalize(text)
    tokens = processor.char_tokenize(normalized)
    ids = processor.text_to_ids(text, vocab)
    
    print(f'Texte :     {text}')
    print(f'Tokens :    {tokens}')
    print(f'IDs :       {ids}')
    print(f'Longueur :  {len(ids)} tokens')
    print()

In [ ]:
# Visualiser la distribution des longueurs de tokens
token_lengths = [len(processor.text_to_ids(t, vocab)) for t in df['transcription']]

fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(token_lengths, bins=30, color='#E91E63', edgecolor='white', alpha=0.8)
ax.set_xlabel('Longueur de la Séquence de Tokens')
ax.set_ylabel('Fréquence')
ax.set_title('Distribution des Longueurs de Séquences de Tokens Après Normalisation')
ax.axvline(np.mean(token_lengths), color='red', linestyle='--', 
           label=f'Moyenne : {np.mean(token_lengths):.0f} tokens')
ax.legend()

plt.tight_layout()
plt.savefig('../results/figures/token_length_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Longueur moyenne des tokens : {np.mean(token_lengths):.1f}')
print(f'Longueur maximale des tokens : {max(token_lengths)}')

## 4. Création du Jeu de Données Annoté

Nous créons maintenant le jeu de données annoté final avec :
- Identifiant unique pour chaque échantillon
- Transcription originale
- Texte normalisé
- Séquence de tokens
- Longueur des tokens

In [ ]:
# Créer le jeu de données annoté
annotated = pd.DataFrame({
    'id': [f'hassaniya_{i:04d}' for i in range(len(df))],
    'original_text': df['transcription'],
    'normalized_text': df['normalized_text'],
    'token_ids': [str(processor.text_to_ids(t, vocab)) for t in df['transcription']],
    'token_length': token_lengths,
    'text_length': df['transcription'].str.len(),
})

# Sauvegarder le jeu de données annoté
annotated.to_csv('../data/metadata.csv', index=False, encoding='utf-8')

print(f'Jeu de données annoté sauvegardé : {len(annotated)} échantillons')
print(f'\nExemples d\'entrées :')
print(annotated[['id', 'normalized_text', 'token_length']].head(10).to_string())

In [ ]:
# Sauvegarder le vocabulaire pour le modèle TTS
import json

vocab_path = '../data/vocab.json'
with open(vocab_path, 'w', encoding='utf-8') as f:
    json.dump(vocab, f, ensure_ascii=False, indent=2)

print(f'Vocabulaire sauvegardé dans {vocab_path}')
print(f'Taille du vocabulaire : {len(vocab)} tokens')

## 5. Analyse de la Qualité de l'Annotation

In [ ]:
# Statistiques résumées
print('=' * 60)
print('     RÉSUMÉ DE L\'ANNOTATION')
print('=' * 60)
print(f'  Échantillons annotés :        {len(annotated)}')
print(f'  Taille du vocabulaire :       {len(vocab)} caractères')
print(f'  Tokens spéciaux :             4 (pad, sos, eos, space)')
print(f'  Caractères arabes :           {len(vocab) - 4}')
print(f'  Textes modifiés par norm. :   {changed_count}')
print(f'  Longueur moy. des tokens :    {np.mean(token_lengths):.1f}')
print(f'  Longueur max. des tokens :    {max(token_lengths)}')
print('=' * 60)
print()
print('Pipeline d\'annotation : Texte brut → Normaliser → Tokeniser → Séquence d\'IDs')
print('Prêt pour le prétraitement (Notebook 03)')

## Conclusion

Dans ce notebook, nous avons construit un pipeline complet d'annotation de texte pour l'arabe hassaniya :

1. **Normalisation** : Standardisation des formes d'Alef, suppression des diacritiques, nettoyage de la ponctuation
2. **Tokenisation** : Tokenisation au niveau des caractères avec marqueurs d'espace
3. **Vocabulaire** : Construction d'une correspondance caractères → identifiants entiers
4. **Jeu de données annoté** : Sauvegardé avec texte original, normalisé et tokenisé

**Observation clé** : La normalisation a réduit notre jeu de caractères, facilitant
l'apprentissage par le modèle TTS de correspondances cohérentes du texte vers la parole.

**Prochaine étape** : Notebook 03 — Prétraitement audio et extraction de caractéristiques.